Scan the directory 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch for JSON files ending with _urls. The filename starts with apkname, followed by version. Save the results to apk_versions_summary.csv in the same directory.

In [186]:
# from pathlib import Path
# import re
# import pandas as pd

# # AA1_first_328_batch
# # 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch
# # 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch
# # 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch
# # 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch
# # 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch
# ROOT_DIR = Path(r"1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA1_first_328_batch")

# OUT_CSV = Path(r"1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA1_first_328_batch\apk_versions_summary.csv") #ROOT_DIR / "apk_versions_summary.csv"

# pat = re.compile(r"^(?P<apk>.+)-(?P<version>\d+)_urls\.json$", re.IGNORECASE)

# rows = []
# bad_files = []

# for p in ROOT_DIR.glob("*_urls.json"):
#     m = pat.match(p.name)
#     if not m:
#         bad_files.append(p.name)
#         continue
#     rows.append({
#         "apk_name": m.group("apk"),
#         "version": m.group("version"),
#         "json_file": p.name,
#         "source": ""
#     })

# df = pd.DataFrame(rows)
# # df["version_num"] = pd.to_numeric(df["version"], errors="coerce")
# # df = df.drop_duplicates(subset=["apk_name", "version"], keep="first")
# df = df.sort_values(["apk_name", "version"], na_position="last").reset_index(drop=True)

# df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

# print("Matched _urls.json:", len(rows))
# print("Unique (apk_name, version):", len(df))
# print("Bad filenames:", len(bad_files))
# if bad_files:
#     print("Examples:", bad_files[:10])
# print("Wrote:", OUT_CSV)

Load 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_apkitself_md.csv. For rows where source is null, use the apkname and version from that row.

Then find the hyperlink associated with the keyword "privacy policy" on https://play.google.com/store/apps/datasafety?id={apkname}&hl=en_US.

Finally, save the results as a CSV file: 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\google_play\google_play_privacy_policy_links.csv

In [ ]:
# {f_position}\{s_position}
f_position = "AA2_second_batch"
s_position = "AA6_sisth_100_batch" # AA6_sisth_100_batch, AA7_seventh_100_batch # AA3_third_100_batch, AA4_forth_100_batch, AA5_fifth_100_batch

In [193]:
from pathlib import Path
import time, random, re, json
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib.parse import urljoin, urlparse
import re
from html import unescape

# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_74_batch\apk_versions_summary_apkitself_md.csv
IN_FILES = Path(rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_apkitself_md.csv")

# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\google_play\google_play_privacy_policy_links.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\google_play\google_play_privacy_policy_links.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\google_play\google_play_privacy_policy_links.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\google_play\google_play_privacy_policy_links.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\google_play\google_play_privacy_policy_links.csv
OUT_DIR = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\google_play")
OUT_CSV = OUT_DIR / "google_play_privacy_policy_links.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# if not IN_FILES:
#     raise FileNotFoundError(f"No *_google_store_summary.csv under: {SEG_DIR}")
df_in = pd.read_csv(IN_FILES)
df_in.head(), df_in.shape

# OUT_CSV = Path(r"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\google_play\google_play_privacy_policy_links.csv")

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 2   com.QuranReading.quranbangla       25   
 3   com.QuranReading.quranbangla       26   
 4        com.RedLineGames.Game49      114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps

In [194]:
df_google_play = df_in[df_in["source"].isna() | (df_in["source"].astype(str).str.strip() == "")].copy()
df_google_play.head(), df_google_play.shape

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 4        com.RedLineGames.Game49      114   
 5        com.RedLineGames.Game49      115   
 6     com.RobotSquid.KingOfCrabs      162   
 
                                     json_file source privacy_url source_file  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json    NaN         NaN         NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json    NaN         NaN         NaN   
 4       com.RedLineGames.Game49-114_urls.json    NaN         NaN         NaN   
 5       com.RedLineGames.Game49-115_urls.json    NaN         NaN         NaN   
 6    com.RobotSquid.KingOfCrabs-162_urls.json    NaN         NaN         NaN   
 
   wayback_url downloaded apkitself_url success  
 0         NaN        NaN           NaN     NaN  
 1         NaN        NaN           NaN     NaN  
 4         NaN        NaN           NaN     NaN  
 5         NaN        NaN         

In [195]:
# df_in = pd.concat([pd.read_csv(p, encoding="utf-8-sig") for p in IN_FILES], ignore_index=True)
if "apk_name" not in df_google_play.columns or "version" not in df_google_play.columns:
    raise ValueError("Input csv must have columns: apk_name, version")

df_google_play["apk_name"] = df_google_play["apk_name"].astype(str).str.strip()
df_google_play["version"] = df_google_play["version"].astype(str).str.strip()
df_google_play = df_google_play.drop_duplicates(subset=["apk_name", "version"]).reset_index(drop=True)

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})
retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))


BAD_HOST_PATTERNS = [
    "policies.google.com/privacy",
    "support.google.com",
    "myaccount.google.com",
    "accounts.google.com",
]

def clean_url(u: str | None) -> str | None:
    if not u:
        return None
    u = unescape(u).strip()
    u = u.replace("\\u003d", "=").replace("\\u0026", "&").replace("\\u002f", "/").replace("\\/", "/")
    u = u.strip('\'" ')
    return u

def is_bad_google_policy_url(u: str | None) -> bool:
    if not u:
        return True
    ul = u.lower()
    return any(p in ul for p in BAD_HOST_PATTERNS)



def is_google_policy_url(href: str) -> bool:
    if not href:
        return True
    h = href.lower()
    return (
        "policies.google.com/privacy" in h
        or "support.google.com" in h
        or "myaccount.google.com" in h
        or "google.com/policies" in h
    )

def extract_privacy_policy_url(html: str) -> str | None:
    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(html, "html.parser")

        # 1) First, look for a parent node containing the fixed prompt text
        target_text_patterns = [
            "developer's privacy policy",
            "developers privacy policy",
            "for more information about collected and shared data"
        ]

        for node in soup.find_all(string=True):
            txt = " ".join(node.strip().lower().split())
            if any(p in txt for p in target_text_patterns):
                parent = node.parent
                if parent:
                    # First, check links under the current node
                    for a in parent.find_all("a", href=True):
                        href = urljoin("https://play.google.com", a["href"].strip())
                        if not is_google_policy_url(href):
                            return href

                    # Then move up one level and check again
                    gp = parent.parent
                    if gp:
                        for a in gp.find_all("a", href=True):
                            href = urljoin("https://play.google.com", a["href"].strip())
                            if not is_google_policy_url(href):
                                return href

        # 2) Fallback: find all links whose text is "privacy policy"
        candidates = []
        for a in soup.find_all("a", href=True):
            text = " ".join(a.get_text(" ", strip=True).lower().split())
            href = urljoin("https://play.google.com", a["href"].strip())

            if "privacy policy" in text:
                candidates.append(href)

        # 3) Prefer non-Google policy links
        for href in candidates:
            if not is_google_policy_url(href):
                return href

        return None

    except Exception:
        # fallback regex
        matches = re.findall(
            r'href="([^"]+)"[^>]*>\s*privacy\s+policy\s*<',
            html,
            flags=re.IGNORECASE
        )
        for href in matches:
            full = urljoin("https://play.google.com", href.strip())
            if not is_google_policy_url(full):
                return full
        return None

def extract_privacy_policy_url_(html: str) -> str | None:
    # 1) First, extract the app-specific privacy policy from script/JSON fields
    patterns = [
        r'"privacyPolicyUrl"\s*:\s*"([^"]+)"',
        r'"privacyPolicy"\s*:\s*"([^"]+)"',
        r'"privacy_policy"\s*:\s*"([^"]+)"',
        r'"privacyPolicyUrl","([^"]+)"',
        r'"privacyPolicy","([^"]+)"',
    ]

    for pat in patterns:
        m = re.search(pat, html, flags=re.IGNORECASE)
        if m:
            u = clean_url(m.group(1))
            if u and u.startswith("http") and not is_bad_google_policy_url(u):
                return u

    # 2) Then scan anchor tags with BeautifulSoup, filtering out Google's own privacy links
    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(html, "html.parser")

        candidates = []

        for a in soup.find_all("a", href=True):
            href = clean_url(urljoin("https://play.google.com", a["href"]))
            text = a.get_text(" ", strip=True).lower()

            if not href or is_bad_google_policy_url(href):
                continue

            # Match by link text
            if "privacy policy" in text:
                candidates.append(href)
                continue

            # Match when the href itself has policy-related semantics
            href_l = href.lower()
            if "privacy" in href_l and "policy" in href_l:
                candidates.append(href)

        # Deduplicate and return the first result
        seen = set()
        dedup = []
        for c in candidates:
            if c not in seen:
                seen.add(c)
                dedup.append(c)

        return dedup[0] if dedup else None

    except Exception:
        return None


def fetch_one(apk: str) -> tuple[str, int | None, str, str]:
    # Check details first, then datasafety
    urls = [
        # f"https://play.google.com/store/apps/details?id={apk}&hl=en_US&gl=US",
        f"https://play.google.com/store/apps/datasafety?id={apk}&hl=en_US&gl=US",
    ]

    last_status = None
    last_err = ""

    for url in urls:
        try:
            resp = session.get(url, timeout=(10, 60))
            last_status = resp.status_code
            if resp.status_code != 200:
                last_err = f"http_{resp.status_code}"
                continue

            resp.encoding = resp.apparent_encoding or resp.encoding
            pp = extract_privacy_policy_url(resp.text)
            if pp:
                return pp, resp.status_code, url, ""

        except Exception as e:
            last_err = f"exception:{type(e).__name__}"

    return "", last_status, "", last_err or "not_found"


rows = []
total = len(df_google_play)

for i, r in df_google_play.iterrows():
    apk = r["apk_name"]
    ver = r["version"]

    privacy_url, status, source_url, err = fetch_one(apk)

    rows.append({
        "apk_name": apk,
        "version": ver,
        "source_url": source_url,
        "privacy_policy_url": privacy_url,
        "fetch_status": status if status is not None else "",
        "error": err,
    })

    if (i + 1) % 20 == 0:
        print(f"Progress: {i+1}/{total}")

    time.sleep(random.uniform(0.8, 1.6))

df_out = pd.DataFrame(rows)
df_out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Wrote:", OUT_CSV)
print("Found privacy_policy_url:", (df_out["privacy_policy_url"].astype(str).str.len() > 0).sum(), "/", len(df_out))

Progress: 20/54
Progress: 40/54
Wrote: 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA2_second_batch\AA5_fifth_100_batch\google_play\google_play_privacy_policy_links.csv
Found privacy_policy_url: 32 / 54


Load 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\google_play\google_play_privacy_policy_links.csv and read the apk_name and privacy_policy_url columns.
Visit privacy_policy_url, download the main privacy policy content as Markdown, save it to 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_google_play\{apkname}, and also save the download results in a CSV file.

In [196]:
from pathlib import Path
import re
import time
import random
from html import unescape

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# =========================
# paths
# =========================
# IN_DIR = Path(r"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch_seg_json")

# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\google_play\google_play_privacy_policy_links.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\google_play\google_play_privacy_policy_links.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\google_play\google_play_privacy_policy_links.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\google_play\google_play_privacy_policy_links.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\google_play\google_play_privacy_policy_links.csv
IN_DIR_Path = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\google_play\google_play_privacy_policy_links.csv")

# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\google_play
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\google_play
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\google_play
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\google_play
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\google_play
OUT_DIR = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\google_play")
OUT_CSV = OUT_DIR / "pp_googleplay_download_results.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# IN_FILES = sorted(IN_DIR.glob("*_privacy_policy_links.csv"))
# if not IN_FILES:
#     raise FileNotFoundError(f"No *_privacy_policy_links.csv found under: {IN_DIR}")

In [197]:
# =========================
# requests session
# =========================
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})

retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)


# =========================
# helpers
# =========================
def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", name)
    name = re.sub(r"\s+", "_", name)
    return name[:200] if name else "unknown_apk"


def html_to_text(html: str) -> str:
    html = re.sub(r"(?is)<script.*?>.*?</script>", " ", html)
    html = re.sub(r"(?is)<style.*?>.*?</style>", " ", html)
    html = re.sub(r"(?i)</p>|<br\s*/?>|</div>|</li>|</tr>|</h[1-6]>", "\n", html)
    html = re.sub(r"(?i)<li[^>]*>", "- ", html)
    text = re.sub(r"(?s)<[^>]+>", " ", html)
    text = unescape(text)
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    return text.strip()


def extract_main_html(html: str) -> str:
    """
    Extract the main content area when possible; if extraction fails, return the original HTML
    """
    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(html, "html.parser")

        # Remove obvious noise
        for tag in soup(["script", "style", "noscript", "svg", "iframe", "footer", "nav"]):
            tag.decompose()

        # Prefer main-content containers
        candidates = []

        for selector in [
            "main",
            "article",
            '[role="main"]',
            ".content",
            ".main-content",
            ".post-content",
            ".entry-content",
            ".page-content",
            ".policy",
            ".privacy-policy",
        ]:
            for node in soup.select(selector):
                txt = node.get_text(" ", strip=True)
                if len(txt) > 500:
                    candidates.append((len(txt), str(node)))

        if candidates:
            candidates.sort(reverse=True)
            return candidates[0][1]

        # Fallback: find the div/section containing the most text
        blocks = []
        for node in soup.find_all(["div", "section"]):
            txt = node.get_text(" ", strip=True)
            if len(txt) > 1000:
                blocks.append((len(txt), str(node)))

        if blocks:
            blocks.sort(reverse=True)
            return blocks[0][1]

        body = soup.body
        if body:
            return str(body)

        return html

    except Exception:
        return html


def html_to_markdown(html: str) -> str:
    """
    Prefer markdownify; fall back to plain text if it fails
    """
    main_html = extract_main_html(html)

    try:
        from markdownify import markdownify as md
        md_text = md(main_html, heading_style="ATX")
        md_text = unescape(md_text)
        md_text = re.sub(r"\n{3,}", "\n\n", md_text).strip()
        return md_text
    except Exception:
        return html_to_text(main_html)


def fetch_and_save_markdown(apk_name: str, version: str, url: str):
    out_path = OUT_DIR / f"{apk_name}_{version}.md"

    try:
        resp = session.get(url, timeout=(10, 90), allow_redirects=True)
        status = resp.status_code

        if status != 200:
            return {
                "success": False,
                "http_status": status,
                "final_url": resp.url if hasattr(resp, "url") else "",
                "saved_path": "",
                "content_length": 0,
                "error": f"http_{status}",
            }

        resp.encoding = resp.apparent_encoding or resp.encoding
        # md_text = html_to_markdown(resp.text)

        # if not md_text or len(md_text.strip()) < 50:
        #     return {
        #         "success": False,
        #         "http_status": status,
        #         "final_url": resp.url,
        #         "saved_path": "",
        #         "content_length": 0,
        #         "error": "content_too_short",
        #     }

        # with open(out_path, "w", encoding="utf-8") as f:
        #     f.write(md_text)

        return {
            "success": True,
            "http_status": status,
            "final_url": resp.url,
            "saved_path": str(out_path),
            # "content_length": len(md_text),
            "error": "",
        }

    except Exception as e:
        return {
            "success": False,
            "http_status": "",
            "final_url": "",
            "saved_path": "",
            "content_length": 0,
            "error": f"{type(e).__name__}: {e}",
        }

In [198]:
# =========================
# load csv
# =========================
# df_in = pd.concat(
#     [pd.read_csv(p, encoding="utf-8-sig") for p in IN_FILES],
#     ignore_index=True
# )
df_in = pd.read_csv(IN_DIR_Path)
df_in.head(), df_in.shape

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 2        com.RedLineGames.Game49      114   
 3        com.RedLineGames.Game49      115   
 4     com.RobotSquid.KingOfCrabs      162   
 
                                           source_url  \
 0  https://play.google.com/store/apps/datasafety?...   
 1  https://play.google.com/store/apps/datasafety?...   
 2  https://play.google.com/store/apps/datasafety?...   
 3  https://play.google.com/store/apps/datasafety?...   
 4  https://play.google.com/store/apps/datasafety?...   
 
                       privacy_policy_url  fetch_status error  
 0  https://poxelstudios.com/privacy.html           200   NaN  
 1  https://poxelstudios.com/privacy.html           200   NaN  
 2       https://www.tap-nation.io/policy           200   NaN  
 3       https://www.tap-nation.io/policy           200   NaN  
 4   https://robotsquid.com/privacypolicy           200   

In [ ]:
# df_google_play_in = df_in[df_in["source"]=="google_play"]
# df_google_play_in.head(), df_google_play_in.shape

In [199]:
required_cols = {"apk_name", "privacy_policy_url"}
missing = required_cols - set(df_in.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df_in["apk_name"] = df_in["apk_name"].astype(str).str.strip()
df_in["privacy_policy_url"] = df_in["privacy_policy_url"].astype(str).str.strip()

# Clean null values
# Logic: non-null value, non-empty string, and string content is not "nan"
mask = (
    df_in["privacy_policy_url"].notna() & 
    (df_in["privacy_policy_url"] != "") & 
    (df_in["privacy_policy_url"] != "nan")
)

df_in = df_in[mask].copy()

# For the same apk_name, keep the first non-empty link
df_in.shape

(32, 6)

In [200]:
# =========================
# run
# =========================
from marshal import version


rows = []
total = len(df_in)

for i, row in df_in.iterrows():
    apk_name = row["apk_name"]
    version = row["version"]
    privacy_policy_url = row["privacy_policy_url"]

    result = fetch_and_save_markdown(apk_name, version, privacy_policy_url)

    rows.append({
        "apk_name": apk_name,
        "version": version,
        "privacy_policy_url": privacy_policy_url,
        "google_play_url": result["final_url"],
        "http_status": result["http_status"],
        "success": result["success"],
        "saved_path": result["saved_path"],
        # "content_length": result["content_length"],
        "error": result["error"],
    })

    if (i + 1) % 20 == 0 or (i + 1) == total:
        print(f"Progress: {i + 1}/{total}")

    time.sleep(random.uniform(0.8, 1.6))


# =========================
# save result csv
# =========================
df_out = pd.DataFrame(rows)
df_out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("Done.")
print("Output CSV:", OUT_CSV)
print("Downloaded:", int(df_out["success"].sum()), "/", len(df_out))

Progress: 20/32
Progress: 32/32
Progress: 40/32
Done.
Output CSV: 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA2_second_batch\AA5_fifth_100_batch\google_play\pp_googleplay_download_results.csv
Downloaded: 26 / 32


Update the table

Then, for rows in apk_versions_summary_waybackmachine where source is null, use the apkname to search for the link on Google Play. Save the results to 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\apkiteself\google_play_privacy_policy_links.csv.

For successful downloads, update the corresponding row in apk_versions_summary.csv—matching apkname and version and where source is apkitself—by changing source to apkitself_waybackmachine.

In [201]:
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_apkitself_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch\apk_versions_summary_apkitself_md.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_apkitself_md.csv"
maintainess_apk_summary_df = pd.read_csv(apk_summary_df_path)
maintainess_apk_summary_df.head(), maintainess_apk_summary_df.shape

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 2   com.QuranReading.quranbangla       25   
 3   com.QuranReading.quranbangla       26   
 4        com.RedLineGames.Game49      114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps

In [202]:
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\google_play\pp_googleplay_download_results.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\google_play\pp_googleplay_download_results.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\google_play\pp_googleplay_download_results.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\google_play\pp_googleplay_download_results.csv
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\google_play\pp_googleplay_download_results.csv
csv_path = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\google_play\pp_googleplay_download_results.csv") 
md_results_df = pd.read_csv(csv_path, dtype={"apk_name": str, "version": str})
md_results_df.head(), md_results_df.shape

(                        apk_name version  \
 0  com.PoxelStudios.CrossyBrakes      27   
 1  com.PoxelStudios.CrossyBrakes      28   
 2        com.RedLineGames.Game49     114   
 3        com.RedLineGames.Game49     115   
 4     com.RobotSquid.KingOfCrabs     162   
 
                       privacy_policy_url  \
 0  https://poxelstudios.com/privacy.html   
 1  https://poxelstudios.com/privacy.html   
 2       https://www.tap-nation.io/policy   
 3       https://www.tap-nation.io/policy   
 4   https://robotsquid.com/privacypolicy   
 
                          google_play_url  http_status  success  \
 0  https://poxelstudios.com/privacy.html        200.0     True   
 1  https://poxelstudios.com/privacy.html        200.0     True   
 2      https://www.tap-nation.io/policy/        200.0     True   
 3      https://www.tap-nation.io/policy/        200.0     True   
 4                                    NaN          NaN    False   
 
                                           saved_pat

In [203]:
# 0. Make a copy first to avoid modifying the original table
left = maintainess_apk_summary_df.copy()
right = md_results_df.copy()
left.head(), right.head()

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 2   com.QuranReading.quranbangla       25   
 3   com.QuranReading.quranbangla       26   
 4        com.RedLineGames.Game49      114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps

In [204]:
# 1. Normalize the data types of key fields
for df in [left, right]:
    df["apk_name"] = df["apk_name"].astype(str).str.strip()
    df["version"] = df["version"].astype(str).str.strip()

In [205]:
# 2. Keep only rows where google_play_url is not empty
src = right.loc[
    right["google_play_url"].notna() &
    (right["google_play_url"].astype(str).str.strip() != ""),
    ["apk_name", "version", "google_play_url", "success"]
].copy()
src.head(), src.shape

(                        apk_name version  \
 0  com.PoxelStudios.CrossyBrakes      27   
 1  com.PoxelStudios.CrossyBrakes      28   
 2        com.RedLineGames.Game49     114   
 3        com.RedLineGames.Game49     115   
 6           com.punjabimatrimony     335   
 
                                      google_play_url  success  
 0              https://poxelstudios.com/privacy.html     True  
 1              https://poxelstudios.com/privacy.html     True  
 2                  https://www.tap-nation.io/policy/     True  
 3                  https://www.tap-nation.io/policy/     True  
 6  https://www.punjabimatrimony.com/privacy-polic...     True  ,
 (30, 4))

In [206]:
# 4. Left join to the target table
merged = left.merge(
    src,
    on=["apk_name", "version"],
    how="left",
    suffixes=("", "_new")
)
merged.head(), merged.shape

(                        apk_name version  \
 0  com.PoxelStudios.CrossyBrakes      27   
 1  com.PoxelStudios.CrossyBrakes      28   
 2   com.QuranReading.quranbangla      25   
 3   com.QuranReading.quranbangla      26   
 4        com.RedLineGames.Game49     114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json                       NaN   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json                       NaN   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json                       NaN   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps/Quran

In [ ]:
# mask = merged["success_new"] == True
# mask.value_counts()

In [207]:
# 6. Update matched rows
mask = merged["success_new"] == True
# merged.loc[mask, "privacy_url"] = merged.loc[mask, "privacy_policy_url"]
merged.loc[mask, "source"] = "google_play"
merged.head(), merged.shape

(                        apk_name version  \
 0  com.PoxelStudios.CrossyBrakes      27   
 1  com.PoxelStudios.CrossyBrakes      28   
 2   com.QuranReading.quranbangla      25   
 3   com.QuranReading.quranbangla      26   
 4        com.RedLineGames.Game49     114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json               google_play   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json               google_play   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json               google_play   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps/Quran

In [ ]:
# # 7. Remove temporary columns
# merged = merged.drop(columns=["privacy_policy_url"])
# merged.head(), merged.shape

In [208]:
# 8. Save back to CSV
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_google_play_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_google_play_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_google_play_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_google_play_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_74_batch\apk_versions_summary_google_play_md.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_google_play_md.csv"
merged.to_csv(apk_summary_df_path, index=False)

Load r"1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_google_play_md.csv". If source is empty, change it to "empty".

In [209]:
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_google_play_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_google_play_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_google_play_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_google_play_md.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_74_batch\apk_versions_summary_google_play_md.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_google_play_md.csv"
maintainess_apk_summary_df = pd.read_csv(apk_summary_df_path)
maintainess_apk_summary_df.head(), maintainess_apk_summary_df.shape

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 2   com.QuranReading.quranbangla       25   
 3   com.QuranReading.quranbangla       26   
 4        com.RedLineGames.Game49      114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json               google_play   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json               google_play   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json               google_play   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps

In [210]:
maintainess_apk_summary_df["source"] = (
    maintainess_apk_summary_df["source"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace("", "empty")
)
maintainess_apk_summary_df.head(), maintainess_apk_summary_df.shape

(                        apk_name  version  \
 0  com.PoxelStudios.CrossyBrakes       27   
 1  com.PoxelStudios.CrossyBrakes       28   
 2   com.QuranReading.quranbangla       25   
 3   com.QuranReading.quranbangla       26   
 4        com.RedLineGames.Game49      114   
 
                                     json_file                    source  \
 0  com.PoxelStudios.CrossyBrakes-27_urls.json               google_play   
 1  com.PoxelStudios.CrossyBrakes-28_urls.json               google_play   
 2   com.QuranReading.quranbangla-25_urls.json  apkitself_waybackmachine   
 3   com.QuranReading.quranbangla-26_urls.json  apkitself_waybackmachine   
 4       com.RedLineGames.Game49-114_urls.json               google_play   
 
                                          privacy_url  \
 0                                                NaN   
 1                                                NaN   
 2  http://www.quranreading.com/apps/Quran-reading...   
 3  http://www.quranreading.com/apps

In [211]:
# 8. Save back to CSV
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_final.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_final.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_final.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_final.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_74_batch\apk_versions_summary_final.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_final.csv"
maintainess_apk_summary_df.to_csv(apk_summary_df_path, index=False)